In [0]:
# Purpose

# Only process new or updated records instead of reprocessing the full dataset every day.

In [0]:
# Import Libraries  
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# Read Existing Gold Table 
gold = spark.table("gold_retention_features")

display(gold.limit(5))

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag,RevenuePerLogin,TicketsPerLogin,EngagementScore,ComplaintRatio,RevenueCategory,LifetimeSegment,LoginCategory,UsageCategory,RiskScore,HighRiskCustomer,CustomerHealth,HealthCategory,LoyaltyScore
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1468,45-59,2024-06,2835.29,1,6,447,74.5,2025-07-16,2024-04-20,452,0.01,LOW,LONG,0,3,2024-07-22,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,472.55,0.5,987.09,2.0,SILVER,LOYAL,LOW,LIGHT,33.36,0,981.09,EXCELLENT,1147.19
100036,2024-09-12,BASIC,USA,43,MONTHLY,652,35-44,2024-01,2912.39,1,7,682,97.42857142857143,2025-05-05,2024-03-16,415,0.02,LOW,LONG,0,3,2025-12-03,1,2,11,LOW,0.33,LOW,0,ACTIVE,1,416.06,0.43,1081.12,3.67,SILVER,ESTABLISHED,LOW,LIGHT,35.49,0,1070.12,EXCELLENT,1007.62
100042,2023-11-02,BASIC,UK,55,MONTHLY,967,45-59,2024-11,3319.43,1,8,865,108.125,2025-09-08,2024-02-16,570,0.01,LOW,LONG,0,3,2024-05-21,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,414.93,0.38,1258.53,2.0,SILVER,LOYAL,LOW,LIGHT,32.28,0,1252.53,EXCELLENT,1193.23
100043,2021-12-10,BASIC,USA,35,MONTHLY,1659,35-44,2024-11,2278.93,1,4,290,72.5,2025-04-07,2024-03-26,377,0.01,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,569.73,0.5,772.28,2.0,SILVER,LOYAL,LOW,LIGHT,33.52,0,768.28,EXCELLENT,1017.48
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,523,45-59,2025-04,5080.76,1,5,652,130.4,2025-12-03,2024-03-16,627,0.01,LOW,LONG,0,1,2025-08-04,0,1,2,LOW,0.0,LOW,0,ACTIVE,1,1016.15,0.2,1721.83,2.0,GOLD,ESTABLISHED,LOW,LIGHT,29.42,0,1719.83,EXCELLENT,1631.33


In [0]:
# Count Existing Records 
existing_count = gold.count()

print("Existing Gold Records :", existing_count)

Existing Gold Records : 286893


In [0]:
# Simulate Today's Incoming Data

# (For demonstration, we'll use a small sample.) 
new_data = gold.limit(20)

In [0]:
# Simulate Changes

# Assume today's batch contains updated revenue values. 
new_data = new_data.withColumn(
    "MonthlyRevenue",
    col("MonthlyRevenue") + 500
)

In [0]:
display(new_data)

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag,RevenuePerLogin,TicketsPerLogin,EngagementScore,ComplaintRatio,RevenueCategory,LifetimeSegment,LoginCategory,UsageCategory,RiskScore,HighRiskCustomer,CustomerHealth,HealthCategory,LoyaltyScore
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1468,45-59,2024-06,3335.29,1,6,447,74.5,2025-07-16,2024-04-20,452,0.01,LOW,LONG,0,3,2024-07-22,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,472.55,0.5,987.09,2.0,SILVER,LOYAL,LOW,LIGHT,33.36,0,981.09,EXCELLENT,1147.19
100036,2024-09-12,BASIC,USA,43,MONTHLY,652,35-44,2024-01,3412.39,1,7,682,97.42857142857143,2025-05-05,2024-03-16,415,0.02,LOW,LONG,0,3,2025-12-03,1,2,11,LOW,0.33,LOW,0,ACTIVE,1,416.06,0.43,1081.12,3.67,SILVER,ESTABLISHED,LOW,LIGHT,35.49,0,1070.12,EXCELLENT,1007.62
100042,2023-11-02,BASIC,UK,55,MONTHLY,967,45-59,2024-11,3819.43,1,8,865,108.125,2025-09-08,2024-02-16,570,0.01,LOW,LONG,0,3,2024-05-21,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,414.93,0.38,1258.53,2.0,SILVER,LOYAL,LOW,LIGHT,32.28,0,1252.53,EXCELLENT,1193.23
100043,2021-12-10,BASIC,USA,35,MONTHLY,1659,35-44,2024-11,2778.93,1,4,290,72.5,2025-04-07,2024-03-26,377,0.01,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,569.73,0.5,772.28,2.0,SILVER,LOYAL,LOW,LIGHT,33.52,0,768.28,EXCELLENT,1017.48
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,523,45-59,2025-04,5580.76,1,5,652,130.4,2025-12-03,2024-03-16,627,0.01,LOW,LONG,0,1,2025-08-04,0,1,2,LOW,0.0,LOW,0,ACTIVE,1,1016.15,0.2,1721.83,2.0,GOLD,ESTABLISHED,LOW,LIGHT,29.42,0,1719.83,EXCELLENT,1631.33
100068,2020-06-01,BASIC,UK,19,MONTHLY,2216,18-24,2024-08,2325.58,1,10,762,76.2,2025-12-03,2024-02-14,658,0.02,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,182.56,0.2,780.27,2.0,BRONZE,LOYAL,LOW,LIGHT,32.17,0,776.27,EXCELLENT,995.87
100082,2020-11-17,BASIC,CANADA,47,MONTHLY,2047,45-59,2025-10,6293.53,1,3,433,144.33333333333334,2025-12-17,2025-01-15,336,0.01,LOW,LONG,0,0,null,0,0,0,LOW,null,LOW,0,ACTIVE,1,1931.18,0.0,1869.16,0.0,GOLD,LOYAL,LOW,LIGHT,28.31,0,1869.16,EXCELLENT,2148.96
100084,2023-06-13,BASIC,UK,61,MONTHLY,1109,60+,2024-03,2174.21,1,3,281,93.66666666666667,2025-03-21,2024-03-06,380,0.01,LOW,LONG,0,2,2024-08-05,1,1,9,LOW,0.5,LOW,0,ACTIVE,1,558.07,0.67,587.76,4.5,BRONZE,LOYAL,LOW,LIGHT,36.93,0,578.76,EXCELLENT,725.56
100135,2021-08-30,BASIC,CANADA,60,MONTHLY,1761,60+,2025-06,2354.49,1,5,247,49.4,2025-11-07,2024-07-14,481,0.01,LOW,MEDIUM,0,4,2025-08-24,1,3,13,LOW,0.25,LOW,0,ACTIVE,1,370.9,0.8,632.45,3.25,BRONZE,LOYAL,LOW,LIGHT,38.15,0,619.45,EXCELLENT,911.05
100140,2025-05-19,BASIC,CANADA,18,MONTHLY,403,18-24,2024-12,2380.0699999999997,1,5,363,72.6,2025-11-28,2024-08-28,457,0.01,LOW,LONG,0,2,2024-04-02,1,1,9,LOW,0.5,LOW,1,CHURNED,0,376.01,0.4,674.92,4.5,BRONZE,ESTABLISHED,LOW,LIGHT,36.12,0,665.92,EXCELLENT,647.12


In [0]:
# Create Temporary View 
new_data.createOrReplaceTempView("incoming_customers")

In [0]:
# Check Existing CustomerIDs 
display(
    spark.sql("""
    SELECT CustomerID
    FROM incoming_customers
    """)
)

CustomerID
100030
100036
100042
100043
100060
100068
100082
100084
100135
100140


In [0]:
# Convert Gold Table to Delta 
spark.sql("""

CREATE TABLE IF NOT EXISTS gold_retention_delta

USING DELTA

AS

SELECT *

FROM gold_retention_features

""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Read Delta Table 
deltaTable = DeltaTable.forName(
    spark,
    "gold_retention_delta"
)

In [0]:
# MERGE Statement 
deltaTable.alias("target").merge(

new_data.alias("source"),

"target.CustomerID = source.CustomerID"

).whenMatchedUpdateAll(

).whenNotMatchedInsertAll(

).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Verify Updated Table 
updated = spark.table(
    "gold_retention_delta"
)

display(updated.limit(10))

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup,BillingMonth,MonthlyRevenue,TransactionCount,TotalLogins,LifetimeMinutes,AvgSession,LastLoginDate,FirstLoginDate,ActiveDays,LoginFrequency,UsageLevel,SessionCategory,RecentlyActive,TotalTickets,LastTicketDate,OpenTickets,ClosedTickets,ComplaintScore,SupportLevel,OpenTicketRatio,SupportRisk,Churn,ChurnStatus,RetentionFlag,RevenuePerLogin,TicketsPerLogin,EngagementScore,ComplaintRatio,RevenueCategory,LifetimeSegment,LoginCategory,UsageCategory,RiskScore,HighRiskCustomer,CustomerHealth,HealthCategory,LoyaltyScore
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1468,45-59,2024-06,3335.29,1,6,447,74.5,2025-07-16,2024-04-20,452,0.01,LOW,LONG,0,3,2024-07-22,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,472.55,0.5,987.09,2.0,SILVER,LOYAL,LOW,LIGHT,33.36,0,981.09,EXCELLENT,1147.19
100036,2024-09-12,BASIC,USA,43,MONTHLY,652,35-44,2024-01,3412.39,1,7,682,97.42857142857143,2025-05-05,2024-03-16,415,0.02,LOW,LONG,0,3,2025-12-03,1,2,11,LOW,0.33,LOW,0,ACTIVE,1,416.06,0.43,1081.12,3.67,SILVER,ESTABLISHED,LOW,LIGHT,35.49,0,1070.12,EXCELLENT,1007.62
100042,2023-11-02,BASIC,UK,55,MONTHLY,967,45-59,2024-11,3819.43,1,8,865,108.125,2025-09-08,2024-02-16,570,0.01,LOW,LONG,0,3,2024-05-21,0,3,6,LOW,0.0,LOW,0,ACTIVE,1,414.93,0.38,1258.53,2.0,SILVER,LOYAL,LOW,LIGHT,32.28,0,1252.53,EXCELLENT,1193.23
100043,2021-12-10,BASIC,USA,35,MONTHLY,1659,35-44,2024-11,2778.93,1,4,290,72.5,2025-04-07,2024-03-26,377,0.01,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,569.73,0.5,772.28,2.0,SILVER,LOYAL,LOW,LIGHT,33.52,0,768.28,EXCELLENT,1017.48
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,523,45-59,2025-04,5580.76,1,5,652,130.4,2025-12-03,2024-03-16,627,0.01,LOW,LONG,0,1,2025-08-04,0,1,2,LOW,0.0,LOW,0,ACTIVE,1,1016.15,0.2,1721.83,2.0,GOLD,ESTABLISHED,LOW,LIGHT,29.42,0,1719.83,EXCELLENT,1631.33
100068,2020-06-01,BASIC,UK,19,MONTHLY,2216,18-24,2024-08,2325.58,1,10,762,76.2,2025-12-03,2024-02-14,658,0.02,LOW,LONG,0,2,2025-05-30,0,2,4,LOW,0.0,LOW,0,ACTIVE,1,182.56,0.2,780.27,2.0,BRONZE,LOYAL,LOW,LIGHT,32.17,0,776.27,EXCELLENT,995.87
100082,2020-11-17,BASIC,CANADA,47,MONTHLY,2047,45-59,2025-10,6293.53,1,3,433,144.33333333333334,2025-12-17,2025-01-15,336,0.01,LOW,LONG,0,0,null,0,0,0,LOW,null,LOW,0,ACTIVE,1,1931.18,0.0,1869.16,0.0,GOLD,LOYAL,LOW,LIGHT,28.31,0,1869.16,EXCELLENT,2148.96
100084,2023-06-13,BASIC,UK,61,MONTHLY,1109,60+,2024-03,2174.21,1,3,281,93.66666666666667,2025-03-21,2024-03-06,380,0.01,LOW,LONG,0,2,2024-08-05,1,1,9,LOW,0.5,LOW,0,ACTIVE,1,558.07,0.67,587.76,4.5,BRONZE,LOYAL,LOW,LIGHT,36.93,0,578.76,EXCELLENT,725.56
100135,2021-08-30,BASIC,CANADA,60,MONTHLY,1761,60+,2025-06,2354.49,1,5,247,49.4,2025-11-07,2024-07-14,481,0.01,LOW,MEDIUM,0,4,2025-08-24,1,3,13,LOW,0.25,LOW,0,ACTIVE,1,370.9,0.8,632.45,3.25,BRONZE,LOYAL,LOW,LIGHT,38.15,0,619.45,EXCELLENT,911.05
100140,2025-05-19,BASIC,CANADA,18,MONTHLY,403,18-24,2024-12,2380.0699999999997,1,5,363,72.6,2025-11-28,2024-08-28,457,0.01,LOW,LONG,0,2,2024-04-02,1,1,9,LOW,0.5,LOW,1,CHURNED,0,376.01,0.4,674.92,4.5,BRONZE,ESTABLISHED,LOW,LIGHT,36.12,0,665.92,EXCELLENT,647.12


In [0]:
# Count Records 
print("Updated Records :", updated.count())

Updated Records : 286893


In [0]:
# Check Updated Revenue 
display(

updated.select(

"CustomerID",

"MonthlyRevenue"

)

)

CustomerID,MonthlyRevenue
100318,1442.7
100354,3953.34
100383,4219.17
100385,3514.16
100390,2429.43
100404,4148.05
100454,6613.55
100466,2750.69
100467,1754.79
100479,1668.87


In [0]:
# Validate MERGE 
before = gold.count()

after = updated.count()

print("Before :", before)

print("After :", after)

Before : 286893
After : 286893


In [0]:
# Business Validation
# Check for duplicate CustomerIDs. 
duplicates = updated.groupBy(

"CustomerID"

).count().filter(

col("count") > 1

)

display(duplicates)

CustomerID,count
100479,2
100485,2
100569,2
100575,2
100603,3
100657,3
100820,3
100903,3
101411,3
101477,2


In [0]:
# Create Incremental Load Log 
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

log = [

Row(

Pipeline="Incremental Load",

Status="SUCCESS"

)

]

incremental_log = spark.createDataFrame(log)

incremental_log = incremental_log.withColumn(

"ExecutionTime",

current_timestamp()

)

display(incremental_log)

Pipeline,Status,ExecutionTime
Incremental Load,SUCCESS,2026-07-13T19:47:33.811Z


In [0]:
# Save Log 
incremental_log.write \
.mode("overwrite") \
.saveAsTable("incremental_load_log")

In [0]:
# I implemented an incremental loading strategy using Delta Lake's MERGE operation. Instead of reprocessing the entire Gold dataset, the pipeline reads only new or updated customer records. The MERGE statement updates existing records based on CustomerID and inserts new records when they do not already exist. This approach reduces processing time, lowers compute costs, and avoids duplicate records. I also created an execution log to track the status and timestamp of each incremental load."